# STEP 2.5 · 공식 통계표 검산 (신뢰도 확보)

SAV 원본을 직접 계산한다. 그런데 **우리 계산값이 KPF 공식
통계표와 다르면?** 보고서 전체가 무너진다.

> 판정 기준: 오차 ±1%p 이내면 '일치'(반올림·결측처리 차이 허용).

In [1]:
import sys; sys.path.insert(0,'..')
import pyreadstat, pandas as pd
df23,_ = pyreadstat.read_sav("../data/raw/journalist_2023.sav")
df25,_ = pyreadstat.read_sav("../data/raw/journalist_2025.sav")

### 검산 1: 생성형 AI 활용률 (공식: 2023=54.3%, 2025=58.1%)

In [2]:
# 2023: q43_9='활용 안함' 표시. NaN이면 활용자.
ai23 = df23['q43_9'].isna().mean()*100
# 2025: q38 (1=활용)
ai25 = (df25['q38']==1).mean()*100
print(f"2023 활용률: 공식 54.3% | 계산 {ai23:.1f}%")
print(f"2025 활용률: 공식 58.1% | 계산 {ai25:.1f}%")

2023 활용률: 공식 54.3% | 계산 54.3%
2025 활용률: 공식 58.1% | 계산 58.1%


### 검산 2: 디지털 피로도 (4~5점 비율, 공식: 2023=38.3%, 2025=45.5%)

In [3]:
fat23 = (df23['q6']>=4).mean()*100
fat25 = (df25['q5']>=4).mean()*100
print(f"2023 피로: 공식 38.3% | 계산 {fat23:.1f}%")
print(f"2025 피로: 공식 45.5% | 계산 {fat25:.1f}%")

2023 피로: 공식 38.3% | 계산 38.3%
2025 피로: 공식 45.5% | 계산 45.5%


### 검산표 만들어 저장

In [4]:
check = pd.DataFrame([
    {'지표':'2023 AI활용률','공식값':54.3,'계산값':round(ai23,1)},
    {'지표':'2025 AI활용률','공식값':58.1,'계산값':round(ai25,1)},
    {'지표':'2023 디지털피로','공식값':38.3,'계산값':round(fat23,1)},
    {'지표':'2025 디지털피로','공식값':45.5,'계산값':round(fat25,1)},
])
check['오차(%p)'] = (check['계산값']-check['공식값']).round(1)
check['판정'] = check['오차(%p)'].abs().le(1.0).map({True:'✓ 일치',False:'✗ 불일치'})
check.to_csv("../outputs/tables/validation_check.csv", index=False, encoding='utf-8-sig')
check

,지표,공식값,계산값,오차(%p),판정
0,2023 AI활용률,54.3,54.3,0.0,✓ 일치
1,2025 AI활용률,58.1,58.1,0.0,✓ 일치
2,2023 디지털피로,38.3,38.3,0.0,✓ 일치
3,2025 디지털피로,45.5,45.5,0.0,✓ 일치


**전부 일치한다.** 
> 보고서 문장: *"원본 데이터 재계산 결과가 공식 통계와 일치함을
> 확인하여 분석의 신뢰성을 확보하였다."*

